# 🌾 Farmer Agro Advisor Chatbot (Google Colab)

This notebook builds a **voice + text** farming chatbot using the **AgroQA Dataset**.

**Features:**
- Voice input (for farmers who cannot type)
- Text input
- Crop-specific advice (Maize, Beans, Cassava, General)
- Saves model for Streamlit deployment

---
### Step 1: Upload your dataset
Run the next cell and upload `AgroQA Dataset.csv` when prompted.

In [ ]:
# Install dependencies
!pip install -q pandas scikit-learn speechrecognition pydub

In [ ]:
from google.colab import files
import os

print('Upload AgroQA Dataset.csv')
uploaded = files.upload()

csv_name = [k for k in uploaded.keys() if k.endswith('.csv')][0]
print(f'Uploaded: {csv_name}')

### Step 2: Copy chatbot engine code
Run this cell to create the chatbot engine (same code used in Streamlit).

In [ ]:
%%writefile chatbot_engine.py
from __future__ import annotations
import pickle
import re
from pathlib import Path
from typing import Optional, Tuple
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DEFAULT_THRESHOLD = 0.12

def clean_text(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s?]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

class AgroChatbot:
    def __init__(self, csv_path: str | Path):
        self.csv_path = Path(csv_path)
        self.df = self._load_dataset(self.csv_path)
        self.vectorizer = TfidfVectorizer(
            lowercase=True, stop_words="english",
            ngram_range=(1, 2), max_features=15000,
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(self.df["search_text"])

    @staticmethod
    def _load_dataset(csv_path: Path) -> pd.DataFrame:
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=["Question", "Answer"]).copy()
        df["Crop"] = df["Crop"].astype(str).str.strip().str.lower()
        df["Question"] = df["Question"].astype(str).str.strip()
        df["Answer"] = df["Answer"].astype(str).str.strip()
        df["search_text"] = (df["Crop"] + " " + df["Question"].map(clean_text)).str.strip()
        return df.reset_index(drop=True)

    def get_answer(self, question: str, crop_filter: Optional[str] = None, threshold: float = DEFAULT_THRESHOLD):
        if not question or not question.strip():
            return ("Please ask a farming question.", None, 0.0, None)
        query = clean_text(question)
        crop = (crop_filter or "all").lower().strip()
        if crop != "all":
            mask = self.df["Crop"] == crop
            subset = self.df[mask] if mask.any() else self.df
            matrix = self.tfidf_matrix[mask.values] if mask.any() else self.tfidf_matrix
            query = f"{crop} {query}"
        else:
            subset, matrix = self.df, self.tfidf_matrix
        query_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vec, matrix).flatten()
        best_local_idx = int(scores.argmax())
        best_score = float(scores[best_local_idx])
        row = subset.iloc[best_local_idx]
        if best_score < threshold:
            return ("No close match found. Try rephrasing.", None, best_score, None)
        return row["Answer"], row["Crop"], best_score, row["Question"]

    def save(self, model_path: str | Path) -> None:
        model_path = Path(model_path)
        model_path.parent.mkdir(parents=True, exist_ok=True)
        with open(model_path, "wb") as handle:
            pickle.dump({"df": self.df, "vectorizer": self.vectorizer,
                         "tfidf_matrix": self.tfidf_matrix, "csv_path": str(self.csv_path)}, handle)

    @classmethod
    def load(cls, model_path: str | Path) -> "AgroChatbot":
        with open(model_path, "rb") as handle:
            payload = pickle.load(handle)
        bot = cls.__new__(cls)
        bot.csv_path = Path(payload.get("csv_path", "AgroQA Dataset.csv"))
        bot.df, bot.vectorizer, bot.tfidf_matrix = payload["df"], payload["vectorizer"], payload["tfidf_matrix"]
        return bot

    @property
    def crop_options(self):
        return ["all"] + sorted(self.df["Crop"].unique().tolist())

    @property
    def stats(self):
        return {"total_qa_pairs": len(self.df), "crops": self.df["Crop"].value_counts().to_dict()}

### Step 3: Load dataset and train the chatbot

In [ ]:
import pandas as pd
from chatbot_engine import AgroChatbot

df = pd.read_csv(csv_name)
print('Dataset shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nSample rows:')
display(df.head())

print('\nQuestions per crop:')
print(df['Crop'].value_counts())

# Build chatbot (TF-IDF index)
bot = AgroChatbot(csv_name)
print('\nChatbot ready! Stats:', bot.stats)

### Step 4: Test with TEXT questions

In [ ]:
test_questions = [
    "When should I harvest beans?",
    "How can I control cassava mosaic disease?",
    "What fertilizer should I use for maize?",
    "How can I prevent soil erosion?",
]

for q in test_questions:
    answer, crop, score, matched = bot.get_answer(q)
    print(f'\nQ: {q}')
    print(f'A: {answer}')
    print(f'Crop: {crop} | Confidence: {score:.2%}')

### Step 5: Test with VOICE input

**Important:** Microphone (`pyaudio`) often fails in Colab. Use **Method A (upload audio)** — it always works.

- **Method A:** Record on your phone, upload the file (.wav, .mp3, .m4a, .webm)
- **Method B:** Live microphone (optional — only if install succeeds)

Uses Google Speech Recognition (free, needs internet).

### Step 5a: Test VOICE (simple — 2 cells only)

**You do NOT need to download any voice dataset.**

Run these 2 cells in order:
1. **Install ffmpeg** (cell below)
2. **Create sample voice + auto test** (creates audio and tests chatbot automatically)

You do **NOT** need to download files to your computer.

In [ ]:
# STEP 1 of 2 — Install ffmpeg (required for voice)
!apt-get update -qq
!apt-get install -y -qq ffmpeg
print("ffmpeg installed OK")

In [ ]:
# STEP 2 of 2 — Create sample voice + test chatbot automatically (NO download needed)
!pip install -q gTTS

import io
import os
import time
import speech_recognition as sr
from pydub import AudioSegment
from gtts import gTTS

os.makedirs("sample_voice", exist_ok=True)
recognizer = sr.Recognizer()

# Pick one farming question to test
test_question = "When should I harvest beans?"
audio_file = "sample_voice/farmer_question_1.mp3"

# Create voice file (retry if internet is slow)
print("Creating sample voice file...")
for attempt in range(3):
    try:
        gTTS(text=test_question, lang="en").save(audio_file)
        print(f"Created: {audio_file}")
        break
    except Exception as e:
        print(f"Attempt {attempt + 1} failed: {e}")
        time.sleep(2)
else:
    raise RuntimeError("Could not create voice file. Check internet connection and run again.")

# Transcribe the audio file
print("\nConverting voice to text...")
with open(audio_file, "rb") as f:
    audio_bytes = f.read()

audio = AudioSegment.from_file(io.BytesIO(audio_bytes), format="mp3")
audio = audio.set_channels(1).set_frame_rate(16000)
wav_io = io.BytesIO()
audio.export(wav_io, format="wav")
wav_io.seek(0)

with sr.AudioFile(wav_io) as source:
    audio_data = recognizer.record(source)

heard = recognizer.recognize_google(audio_data)
print(f'Voice heard: "{heard}"')

# Get chatbot answer
answer, crop, score, matched = bot.get_answer(heard)
print(f"\nChatbot Answer: {answer}")
print(f"Crop: {crop} | Confidence: {score:.2%}")
print("\nSUCCESS — Voice test completed!")

In [ ]:
# OPTIONAL — Upload your own voice file (only if you want to test manually)
import io
import speech_recognition as sr
from pydub import AudioSegment
from google.colab import files

recognizer = sr.Recognizer()

def voice_upload_and_answer(crop_filter="all"):
    print("Upload your voice recording (.wav, .mp3, .m4a, .webm, .ogg)\n")
    uploaded = files.upload()
    if not uploaded:
        print("No file uploaded.")
        return

    filename = list(uploaded.keys())[0]
    ext = filename.rsplit(".", 1)[-1].lower()
    audio = AudioSegment.from_file(io.BytesIO(uploaded[filename]), format=ext)
    audio = audio.set_channels(1).set_frame_rate(16000)
    wav_io = io.BytesIO()
    audio.export(wav_io, format="wav")
    wav_io.seek(0)

    with sr.AudioFile(wav_io) as source:
        audio_data = recognizer.record(source)

    text = recognizer.recognize_google(audio_data)
    print(f'You said: "{text}"')
    answer, crop, score, _ = bot.get_answer(text, crop_filter=crop_filter)
    print(f"\nAnswer: {answer}")
    print(f"Crop: {crop} | Confidence: {score:.2%}")

# Uncomment next line ONLY if you want to upload your own audio:
# voice_upload_and_answer()

### Step 6: Interactive chat loop (text or voice)

In [ ]:
def chat_loop():
    print('Farmer Agro Advisor')
    print('  - type a question (text)')
    print('  - type "voice" to upload an audio file')
    print('  - type "quit" to exit\n')
    while True:
        user = input('You: ').strip()
        if user.lower() in ('quit', 'exit', 'q'):
            print('Goodbye!')
            break
        if user.lower() == 'voice':
            voice_upload_and_answer()
            continue
        answer, crop, score, _ = bot.get_answer(user)
        print(f'Advisor: {answer}')
        if crop:
            print(f'  (Crop: {crop}, confidence: {score:.2%})\n')

# Uncomment to start interactive chat:
# chat_loop()

### Step 7: Save model for Streamlit deployment

In [ ]:
import os
os.makedirs('models', exist_ok=True)
model_path = 'models/agro_chatbot.pkl'
bot.save(model_path)
print(f'Model saved to {model_path}')

### Step 8: Download files for Streamlit deployment
Download these and upload to GitHub:
- `AgroQA Dataset.csv`
- `app.py`
- `chatbot_engine.py`
- `voice_utils.py`
- `requirements.txt`
- `models/agro_chatbot.pkl`

In [ ]:
# Create voice_utils.py for Streamlit
%%writefile voice_utils.py
from __future__ import annotations
import tempfile
from pathlib import Path
from typing import Optional
import speech_recognition as sr

def transcribe_audio_file(audio_bytes: bytes, language: str = "en-US"):
    if not audio_bytes:
        return None, "No audio received."
    recognizer = sr.Recognizer()
    try:
        from pydub import AudioSegment
    except ImportError:
        return None, "pydub is required."
    temp_dir = Path(tempfile.gettempdir())
    input_path = temp_dir / "farmer_input_audio"
    wav_path = temp_dir / "farmer_input_audio.wav"
    audio_segment = None
    last_error = None
    for suffix in (".webm", ".wav", ".ogg", ".mp3", ".m4a"):
        try:
            candidate = input_path.with_suffix(suffix)
            candidate.write_bytes(audio_bytes)
            audio_segment = AudioSegment.from_file(str(candidate))
            break
        except Exception as exc:
            last_error = exc
    if audio_segment is None:
        return None, f"Could not read audio: {last_error}"
    audio_segment = audio_segment.set_channels(1).set_frame_rate(16000)
    audio_segment.export(str(wav_path), format="wav")
    with sr.AudioFile(str(wav_path)) as source:
        recognizer.adjust_for_ambient_noise(source, duration=0.3)
        audio_data = recognizer.record(source)
    try:
        return recognizer.recognize_google(audio_data, language=language).strip(), None
    except sr.UnknownValueError:
        return None, "Could not understand the voice."
    except sr.RequestError as exc:
        return None, f"Speech service error: {exc}"

In [ ]:
# Create app.py for Streamlit
%%writefile app.py
# Copy the app.py content from your project folder.
# Or download app.py from the files created on your computer.
print('Upload app.py from your local project folder to GitHub for deployment.')

In [ ]:
from google.colab import files

# Download model
files.download('models/agro_chatbot.pkl')

# Download helper files
for f in ['chatbot_engine.py', 'voice_utils.py']:
    if os.path.exists(f):
        files.download(f)

print('Download complete! Now push all files to GitHub and deploy on share.streamlit.io')